# OTT Content Trend Analysis (Netflix catalog)

Dataset: Netflix Movies and TV Shows (Kaggle, Shivam Bansal). 8,807 titles, data till Sept 2021.

**Sawal jo mujhe solve karna hai:** Netflix ke catalog mein kya trend chal raha hai - genre, country, release year aur movie length ke hisaab se? Isse ek content team ko kya decision milega?

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3

sns.set_style("whitegrid")
df = pd.read_csv("netflix_titles.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isnull().sum())
print(df['type'].value_counts())

**Pehli nazar mein:** director mein sabse zyada missing hai (2634), country mein 831 aur cast mein 825. date_added mein sirf 10 missing hain. Director aur cast ka mujhe trend analysis mein kaam nahi hai, to unko chhod diya.

In [ ]:
# --- Cleaning ---
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Not Rated')

# date_added mein extra space tha, strip karna pada
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added'] = df['date_added'].dt.year

# duration: movie ke liye "90 min", show ke liye "2 Seasons" - number alag nikala
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float)

# ek title ke kai country/genre hote hain, simple rakhne ke liye pehla wala le raha hoon
df['main_country'] = df['country'].str.split(',').str[0].str.strip()
df['main_genre'] = df['listed_in'].str.split(',').str[0].str.strip()

print("duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
df[['title','type','main_country','main_genre','year_added','duration_num']].head()

**Cleaning notes:** duplicates 0 mile. Pehla country/genre lene ka matlab hai multi-country co-productions ek hi country mein count hongi - ye is analysis ki limitation hai, report mein bhi likhunga.

In [ ]:
# Chart 1: Movies vs TV Shows
sns.countplot(data=df, x='type', color='steelblue')
plt.title("Catalog mein Movies vs TV Shows")
plt.show()
print(df['type'].value_counts(normalize=True).round(3))

**Chart 1:** Catalog ka lagbhag 70% Movies hai (6131) aur 30% TV Shows (2676).

In [ ]:
# Chart 2: Release year ke hisaab se count (2010 ke baad)
yearly = (df[df['release_year'] >= 2010]
          .groupby(['release_year', 'type']).size().reset_index(name='count'))
sns.lineplot(data=yearly, x='release_year', y='count', hue='type', marker='o')
plt.title("Titles by release year (2010 onwards)")
plt.show()

**Chart 2:** Movies 2017-18 ke aas-paas peak par gayi aur uske baad girti gayi. TV Shows 2020 tak badhte rahe. Note: 2021 ka data adhura hai (dataset Sept 2021 tak ka hai), isliye wo girawat poori sach nahi.

In [ ]:
# Chart 3: Top 10 genres
top_genres = df['main_genre'].value_counts().head(10)
sns.barplot(x=top_genres.values, y=top_genres.index, color='steelblue')
plt.title("Top 10 genres (main genre)")
plt.xlabel("Titles")
plt.show()

**Chart 3:** Dramas (1600) aur Comedies (1210) clear top par hain. Teesre number par Action & Adventure aur Documentaries lagbhag barabar hain.

In [ ]:
# Chart 4: genre mix heatmap
top5 = df['main_genre'].value_counts().head(5).index
mix = df[df['main_genre'].isin(top5) & (df['release_year'] >= 2010)]
pivot = pd.crosstab(mix['release_year'], mix['main_genre'])
plt.figure(figsize=(9, 6))
sns.heatmap(pivot, cmap="YlOrRd", annot=True, fmt='d')
plt.title("Top 5 genres by release year")
plt.show()

**Chart 4:** Dramas aur Comedies dono 2018 mein peak par the (216 aur 138) aur uske baad girte gaye. Lekin International TV Shows 2018 ke 94 se badhke 2020 mein 115 ho gaye - jab baaki genres neeche ja rahe the tab ye upar gaya. 2021 adhura saal hai.

In [ ]:
# Chart 5: Top 10 countries
top_c = df[df['main_country'] != 'Unknown']['main_country'].value_counts().head(10)
sns.barplot(x=top_c.values, y=top_c.index, color='indianred')
plt.title("Top 10 content producing countries")
plt.xlabel("Titles")
plt.show()

**Chart 5:** US (3211) pehle number par hai aur India (1008) doosre par, UK teesre (628). India ka US ke baad hona mere liye surprise tha.

In [ ]:
# Chart 6: Movie duration trend
movies = df[df['type'] == 'Movie']
avg_dur = movies.groupby('release_year')['duration_num'].mean().reset_index()
avg_dur = avg_dur[avg_dur['release_year'] >= 2000]
sns.lineplot(data=avg_dur, x='release_year', y='duration_num', marker='o')
plt.title("Average movie length (minutes) by release year")
plt.show()

**Chart 6:** 2000-2002 mein average movie lagbhag 115-119 min ki thi, 2020 tak ye 92 min ke paas aa gayi. Yani movies chhoti ho rahi hain.

In [ ]:
# Chart 7 (bonus): ratings
sns.countplot(data=df, y='rating', order=df['rating'].value_counts().index, color='seagreen')
plt.title("Content ratings")
plt.show()

**Chart 7:** TV-MA (3207) aur TV-14 (2160) mil ke catalog ka lagbhag 61% hain - matlab zyadatar content adults/teens ke liye hai, kids ka share chhota hai.

## SQL part
Same data ko SQLite mein daal ke core numbers SQL se nikaal raha hoon.

In [ ]:
conn = sqlite3.connect("netflix.db")
df[['type','title','main_country','release_year','year_added','rating','main_genre','duration_num']] \
    .to_sql("titles", conn, index=False, if_exists="replace")

In [ ]:
q1 = """
SELECT main_genre, COUNT(*) AS total
FROM titles
GROUP BY main_genre
ORDER BY total DESC
LIMIT 10;
"""
pd.read_sql(q1, conn)

In [ ]:
q2 = """
SELECT year_added, type, COUNT(*) AS added
FROM titles
WHERE year_added IS NOT NULL
GROUP BY year_added, type
ORDER BY year_added;
"""
pd.read_sql(q2, conn)

In [ ]:
q3 = """
SELECT main_country, COUNT(*) AS total
FROM titles
WHERE main_country != 'Unknown'
GROUP BY main_country
HAVING COUNT(*) > 50
ORDER BY total DESC;
"""
pd.read_sql(q3, conn)

In [ ]:
q4 = """
SELECT release_year, ROUND(AVG(duration_num), 1) AS avg_minutes
FROM titles
WHERE type = 'Movie' AND release_year >= 2000
GROUP BY release_year
ORDER BY release_year;
"""
pd.read_sql(q4, conn)

In [ ]:
# Excel pivot ke liye clean file
df.to_csv("netflix_clean.csv", index=False)